# 02 - Scanned PDF NER (Task 3)
Pipeline: scanned PDF → PyMuPDF + Tesseract OCR (Spanish) → extracted text → GLiNER (small + merge)
→ strict/relaxed evaluation. Only 10 scanned PDFs are available (test-0001 … test-0010).

In [1]:
import os, sys, warnings
from pathlib import Path

os.environ["HF_HUB_DISABLE_XET"] = "1"                 # evitar la descarga que se colgaba
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"    # ocultar aviso de symlinks
os.environ["TESSDATA_PREFIX"] = r"C:\Users\alexa\tessdata"
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

sys.path.insert(0, str(Path('..').resolve()))          # importar 'ner'

%load_ext autoreload
%autoreload 2

In [2]:
import difflib
from ner.evaluation import evaluate_ner_strict, evaluate_ner_relaxed, ENTITY_LABELS
from ner.inference import load_gliner_model, predict_dataset, merge_dataset
from ner.pdf import extract_text_from_pdf, extract_pdf_dataset
from ner.utils import load_jsonl, save_responses_to_jsonl

VALIDATION_PATH = '../data/plain/validation.jsonl'
PDF_DIR = '../data/pdf_scanned'
LABELS = list(ENTITY_LABELS)

gold_samples = load_jsonl(VALIDATION_PATH)
print(len(gold_samples), 'gold documents')

50 gold documents


In [3]:
pdf_path = f'{PDF_DIR}/test-0001_scanned.pdf'

no_ocr = extract_text_from_pdf(pdf_path, ocr=False)     # solo capa de texto vacía
with_ocr = extract_text_from_pdf(pdf_path, ocr=True)    # Tesseract sobre la imagen

print('Characters WITHOUT OCR:', len(no_ocr))
print('Characters WITH OCR   :', len(with_ocr), '| gold:', len(gold_samples[0]['text']))
print('\n', with_ocr[:400])

Characters WITHOUT OCR: 0
Characters WITH OCR   : 4089 | gold: 4091

 Servicio de Procedimientos Públicos de la Región Central informa de que Patricia Giménez Domínguez, con documento de identidad 12082063W y domicilio habitual en Avenida de Aragón, 102, 2-B, 17001 Girona, presentó el 2018-04-09 una solicitud relacionada con la inscripción en un registro administrativo sectorial. La petición se registró al sistema con el número ASS-2018-43226, y el justificante de e


In [7]:
scanned = extract_pdf_dataset(PDF_DIR, gold_samples, mode='blocks', ocr=True)
print(len(scanned), 'scanned PDFs extracted\n')

for doc, gold in zip(scanned, gold_samples):
    # autojunk=False: compara carácter a carácter sin el atajo que falla con textos largos
    matcher = difflib.SequenceMatcher(None, gold['text'], doc['text'], autojunk=False)
    # Fragmentos distintos: (lo que pone el gold, lo que ha leído el OCR)
    diffs = [(gold['text'][a:b], doc['text'][c:d])
             for tag, a, b, c, d in matcher.get_opcodes() if tag != 'equal']
    print(f"{doc['id']}  similarity={matcher.ratio():.4f}  identical={doc['text'] == gold['text']}  "
          f"differences (gold -> OCR): {diffs[:5]}")

PyMuPDF:   0%|          | 0/50 [00:00<?, ?it/s]

No PDF for test-0011: stopping at 10 documents.
10 scanned PDFs extracted

test-0001  similarity=0.9994  identical=False  differences (gold -> OCR): [('W', 'VV'), (' ', ''), (' ', '')]
test-0002  similarity=0.9965  identical=False  differences (gold -> OCR): [('á', 'à'), ('·', '.'), ('á', 'à'), ('I', 'i'), ('á', 'à')]
test-0003  similarity=0.9996  identical=False  differences (gold -> OCR): [('è', 'e'), (' ', '')]
test-0004  similarity=0.9981  identical=False  differences (gold -> OCR): [('i', '¡'), (' ', ''), ('·', ' '), ('ç', 'c'), ('è', 'é')]
test-0005  similarity=0.9992  identical=False  differences (gold -> OCR): [('.', ','), ('I', 'i'), (' ', ''), (' ', '')]
test-0006  similarity=0.9977  identical=False  differences (gold -> OCR): [('ò', 'ó'), ('G', '6'), ('·', '-'), ('3', '8'), ('A', '4')]
test-0007  similarity=0.9976  identical=False  differences (gold -> OCR): [('·', '-'), ('i', '¡'), ('ç', 'g'), ('á', 'à'), ('·', '-')]
test-0008  similarity=0.9988  identical=False  difference

**OCR quality (PyMuPDF + Tesseract, 300 dpi):**
- Without OCR the scanned PDFs have **0 characters** (they are only images).
- Some documents are in **Catalan**. With only the Spanish model (`spa`) the OCR failed on Catalan
  characters (`ç`, `·`, `à`, `è`). Using `spa+cat` the similarity with the original text is
  **≥ 0.997 for all 10 documents**.
- Note: Python's `difflib` with default settings (autojunk) gave a wrong similarity (0.76) for
  test-0006 because it could not align a long paragraph; we use `autojunk=False`.
- Remaining OCR errors fall partly **inside entities**: `12082063W` → `12082063VV` (ID card),
  `G` → `6`, `3` → `8`, `A` → `4` (IDs and registration numbers), plus missing spaces that shift
  the character offsets. These errors will affect the strict evaluation and also the relaxed one,
  because the entity text changes.

In [5]:
model = load_gliner_model()

OUT_PATH = '../results/pdf_scanned/gliner_small_merge_ocr.jsonl'
preds = predict_dataset(model, scanned, LABELS, threshold=0.5)
save_responses_to_jsonl(merge_dataset(preds), OUT_PATH)

strict = evaluate_ner_strict(VALIDATION_PATH, OUT_PATH)
relaxed = evaluate_ner_relaxed(VALIDATION_PATH, OUT_PATH)

W1112 17:40:52.314000 17500 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

GLiNER:   0%|          | 0/10 [00:00<?, ?it/s]


STRICT EVALUATION
Micro -> P: 0.66 | R: 0.70 | F1: 0.68 | TP: 286 | FP: 148 | FN: 124

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------------------------------------------------------------
PERSON_NAME                 0.68   0.70   0.69      70      33      30
ID_CARD_NUMBER              0.67   0.72   0.70      29      14      11
ADDRESS                     0.57   0.60   0.59      24      18      16
REGISTRATION_NUMBER         0.72   0.66   0.69      33      13      17
DATE                        0.78   0.78   0.78      62      18      18
AMOUNT                      0.54   0.62   0.58      25      21      15
ORGANIZATION                0.58   0.72   0.64      43      31      17

RELAXED EVALUATION - OCCURRENCES
Micro -> P: 0.78 | R: 0.83 | F1: 0.81 | TP: 340 | FP: 94 | FN: 70

ENTITY TYPE                    P      R     F1      TP      FP      FN
-------------------------------------------------------------------------------

In [6]:
plain_preds = load_jsonl('../results/plain/gliner_small_merge.jsonl', n=10)
PLAIN10_PATH = '../results/pdf_scanned/plain_first10_reference.jsonl'
save_responses_to_jsonl(plain_preds, PLAIN10_PATH)

print('########## PLAIN TEXT (same 10 documents) ##########')
evaluate_ner_strict(VALIDATION_PATH, PLAIN10_PATH)
evaluate_ner_relaxed(VALIDATION_PATH, PLAIN10_PATH);

########## PLAIN TEXT (same 10 documents) ##########

STRICT EVALUATION
Micro -> P: 0.87 | R: 0.92 | F1: 0.90 | TP: 379 | FP: 56 | FN: 31

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------------------------------------------------------------
PERSON_NAME                 0.97   1.00   0.99     100       3       0
ID_CARD_NUMBER              0.79   0.82   0.80      33       9       7
ADDRESS                     0.65   0.70   0.67      28      15      12
REGISTRATION_NUMBER         0.96   0.92   0.94      46       2       4
DATE                        1.00   1.00   1.00      80       0       0
AMOUNT                      0.72   0.82   0.77      33      13       7
ORGANIZATION                0.81   0.98   0.89      59      14       1

RELAXED EVALUATION - OCCURRENCES
Micro -> P: 0.87 | R: 0.92 | F1: 0.90 | TP: 379 | FP: 56 | FN: 31

ENTITY TYPE                    P      R     F1      TP      FP      FN
----------------------------

{'compared_samples': 10,
 'occurrences': {'micro': {'precision': 0.871264367816092,
   'recall': 0.9243902439024391,
   'f1': 0.8970414201183432,
   'true_positives': 379,
   'false_positives': 56,
   'false_negatives': 31},
  'per_entity_type': {'PERSON_NAME': {'precision': 0.970873786407767,
    'recall': 1.0,
    'f1': 0.9852216748768473,
    'true_positives': 100,
    'false_positives': 3,
    'false_negatives': 0},
   'ID_CARD_NUMBER': {'precision': 0.7857142857142857,
    'recall': 0.825,
    'f1': 0.8048780487804876,
    'true_positives': 33,
    'false_positives': 9,
    'false_negatives': 7},
   'ADDRESS': {'precision': 0.6511627906976745,
    'recall': 0.7,
    'f1': 0.674698795180723,
    'true_positives': 28,
    'false_positives': 15,
    'false_negatives': 12},
   'REGISTRATION_NUMBER': {'precision': 0.9583333333333334,
    'recall': 0.92,
    'f1': 0.9387755102040817,
    'true_positives': 46,
    'false_positives': 2,
    'false_negatives': 4},
   'DATE': {'precision': 

### Task 3 — Results (same 10 documents)

| Input | Extraction | Strict F1 | Relaxed occ. F1 | Relaxed unique F1 |
|---|---|---|---|---|
| Plain text | — | 0.90 | 0.90 | 0.88 |
| Scanned PDF | PyMuPDF + Tesseract OCR (spa+cat) | 0.68 | 0.81 | 0.78 |

**For entity type (relaxed occurrences F1, plain → OCR):** PERSON_NAME 0.99 → 0.99,
AMOUNT 0.77 → 0.77, REGISTRATION_NUMBER 0.94 → 0.92, ID_CARD_NUMBER 0.80 → 0.77,
DATE 1.00 → 0.94, ADDRESS 0.67 → 0.56, **ORGANIZATION 0.89 → 0.49**.

- **The strict score drops the most (0.90 → 0.68).** OCR sometimes misses a space, and every missing
  character shifts the offsets of the rest of the document, so many correctly found entities no
  longer match the gold spans.
- **Unlike the clean PDFs, the relaxed score also drops (0.90 → 0.81)**, because here the text itself
  changes: one wrong character inside an entity makes it a false positive and a false negative
  (e.g. `12082063W` → `12082063VV`).
- **ORGANIZATION is the most affected type.** Many organization names are in Catalan and contain
  accents (`Econòmica`, `Segrià`) that the OCR reads differently (`ò` → `ó`, `à` → `á`).
  Curiously, for this type strict (0.64) is higher than relaxed (0.49): replacing one accented
  letter does not change the length, so when the offsets are not yet shifted the span still matches
  the gold in strict evaluation even though the text is wrong.
- PERSON_NAME and AMOUNT are not affected by the OCR errors.
- **Conclusion:** even with ≥ 0.997 character similarity, small OCR errors have a big impact on NER
  evaluation, because they fall inside entities and shift the offsets.